In [0]:
from pyspark.sql.functions import format_number
from pyspark.sql import functions as F

In [0]:
import logging
from datetime import datetime

# Set up logging
logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

# Load from Silver Delta table
logger.info("Loading silver table...")

df_silver = spark.read.table("workspace.default.ppr_silver")

logger.info(f"Loaded {df_silver.count()} records from silver table")
df_silver.printSchema()


In [0]:
# Cell - Build Gold table
logger.info("Building Gold table...")

df_gold = df_silver \
    .withColumn("price_band",
        F.when(F.col("price") < 200000, "Under 200k")
        .when(F.col("price") < 400000, "200k-400k")
        .when(F.col("price") < 600000, "400k-600k")
        .otherwise("Over 600k")
    ) \
    .withColumn("decade",
        F.when(F.col("year") < 2015, "2010-2014")
        .when(F.col("year") < 2020, "2015-2019")
        .otherwise("2020-present")
    ) \
    .withColumn("is_dublin", 
        F.when(F.col("county") == "Dublin", True)
        .otherwise(False)
    ) \
    .withColumn("is_new_build",
        F.when(F.col("property_type").contains("New"), True)
        .otherwise(False)
    )

# Save Gold table
df_gold.write \
    .format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("workspace.default.ppr_gold")

logger.info(f"Gold table saved with {df_gold.count()} records")
df_gold.printSchema()

In [0]:
%skip
# Verify Gold table
df_check = spark.read.table("workspace.default.ppr_gold")

print(f"Total records: {df_check.count()}")
print(f"\nPrice band distribution:")
display(df_check.groupBy("price_band").count().orderBy("price_band"))

print(f"\nDecade distribution:")
display(df_check.groupBy("decade").count().orderBy("decade"))

In [0]:
%skip
# Avg price by county and year
df_trands = df_silver \
    .groupBy("county", "year") \
    .agg(
        F.round(F.avg("price"), 2).alias("avg_price"),
        F.round(F.median("price"), 2).alias("median_price"),
        F.count("price").alias("total_sales"),
        F.round(F.min("price"), 2).alias("min_price"),
        F.round(F.max("price"), 2).alias("max_price"),
        F.round(F.stddev("price"),2).alias("std_price"),
    ) \
    .withColumn("avg_price", F.format_number("avg_price", 2)) \
    .withColumn("median_price", F.format_number("median_price", 2)) \
    .withColumn("min_price", F.format_number("min_price", 2)) \
    .withColumn("max_price", F.format_number("max_price", 2)) \
    .withColumn("std_price", F.format_number("std_price", 2)) \
    .orderBy("county", "year")

display(df_trands)

In [0]:
%skip
#Save as Delta Table
df_trands.write \
    .format("delta") \
    .mode("overwrite") \
    .saveAsTable("workspace.default.ppr_gold")

# After cleaning
logger.info(f"Gold records after aggregation: {df_trands.count()}")

# After saving Delta table
logger.info("Gold table saved: workspace.default.ppr_gold")
logger.info("Pipeline completed successfully")

In [0]:
%skip
# Dublin price growth story
df_dublin = df_silver \
    .filter(F.col("county") == "Dublin") \
    .groupBy("year") \
    .agg(
        F.round(F.median("price"), 2).alias("median_price"),
        F.count("price").alias("total_sales")
    ) \
    .withColumn("median_price", F.format_number("median_price", 2)) \
    .withColumn("year", F.col("year").cast("int")) \
    .orderBy("year")

display(df_dublin)